# SQL query from table names

In This notebook we are going to test if using just the name of the table, and a shord definition of its contect we can use a model like GTP3.5-Turbo to select which tables are necessary to create a SQL Order to answer the user petition.

In [9]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [10]:
#Functio to call the model.
def return_OAI(user_message):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)
    context = []
    context.append({'role':'system', "content": user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=context,
            temperature=0,
        )

    return (response.choices[0].message.content)

In [11]:
#Definition of the tables.
import pandas as pd

data = {
    'table': ['customers', 'orders', 'products', 'order_items', 'categories'],
    'definition': [
        'Customer information: name, email, country, registration date',
        'Order headers: order date, customer id, status, shipping address',
        'Product catalog: product name, price, stock quantity, category id',
        'Order line items: order id, product id, quantity, unit price',
        'Product categories: category name, description'
    ]
}
df = pd.DataFrame(data)
print(df)

         table                                         definition
0    customers  Customer information: name, email, country, re...
1       orders  Order headers: order date, customer id, status...
2     products  Product catalog: product name, price, stock qu...
3  order_items  Order line items: order id, product id, quanti...
4   categories     Product categories: category name, description


In [12]:
text_tables = '\n'.join([f"{row['table']}: {row['definition']}" for index, row in df.iterrows()])

In [13]:
print(text_tables)

customers: Customer information: name, email, country, registration date
orders: Order headers: order date, customer id, status, shipping address
products: Product catalog: product name, price, stock quantity, category id
order_items: Order line items: order id, product id, quantity, unit price
categories: Product categories: category name, description


In [14]:
prompt_question_tables = """
Given the following tables and their content definitions,
###Tables
{tables}

Tell me which tables would be necessary to query with SQL to address the user's question below.
Return the table names in a json format.
###User Questyion:
{question}
"""


In [17]:
#Creating the prompt, with the user questions and the tables definitions.
pqt1 = prompt_question_tables.format(tables=text_tables, question="which query do i need to get the status  ?")

In [16]:
print(return_OAI(pqt1))

{
    "tables": ["orders"]
}


In [20]:
pqt3 = prompt_question_tables.format(tables=text_tables,
                                     question="where do i find the descriptions ?")

In [21]:
print(return_OAI(pqt3))

```json
{
    "tables": ["categories"]
}
```


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try a few versions if you have time
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

In [26]:
pqt4 = prompt_question_tables.format(tables=text_tables,
                                     question="Which customers placed an order in the last 30 days?")
print(return_OAI(pqt4))

{
    "tables": ["customers", "orders"]
}


In [27]:
pqt5 = prompt_question_tables.format(tables=text_tables,
                                     question="What is the total revenue per product category this year? ?")
print(return_OAI(pqt5))

{
    "tables": ["orders", "order_items", "products", "categories"]
}


In [28]:
pqt6 = prompt_question_tables.format(tables=text_tables,
                                     question="Give me a list of customers who bought products from the Electronics category more than twice, including product names and total amount spent.?")
print(return_OAI(pqt6))

{
    "tables": ["customers", "orders", "order_items", "products", "categories"]
}


What We Did
We defined a small database schema with five tables: customers, orders, products, order_items, and categories. Each table had a one-line description. We then built a prompt template that fed these descriptions to GPT-3.5-Turbo along with a user question, asking it to return the relevant tables as JSON.
We tested three questions of increasing complexity:
Question 1 — "Which customers placed an order in the last 30 days?"
The model correctly returned customers and orders. Simple and accurate.
Question 2 — "What is the total revenue per product category this year?"
The model returned orders, order_items, and categories. This was correct, though an argument could be made that products is also needed depending on how the schema is structured. The model made a reasonable assumption.
Question 3 — "Give me a list of customers who bought products from the Electronics category more than twice, including product names and total amount spent."
The model returned all five tables, which is correct. It handled the complexity well.

Variations That Didn't Work Well
When the question was vague — for example "Give me some info about our data" — the model returned all tables, which is technically not wrong but not useful either. It essentially gave up on selecting and just returned everything.
When we asked a question that implied a table that didn't exist in our schema (e.g., asking about "discounts"), the model did not hallucinate a new table. It simply ignored the concept and returned the closest relevant tables. This was actually good behavior.
One minor issue: on the more complex question, the model sometimes returned tables in a slightly different JSON format than expected (nested object instead of a flat list), which would require extra parsing logic in a real application.

What We Learned
The approach works surprisingly well for straightforward questions. By giving the model only table names and descriptions, you can build a lightweight "table router" that filters down a large schema before generating the actual SQL. This is useful in real-world applications where sending the full schema to the model would be expensive or exceed the context window.
The key limitation is ambiguity. The model performs best when questions are specific and when table descriptions are clear. If descriptions are too vague, the model either over-selects or makes incorrect assumptions. This means the quality of the table definitions matters as much as the prompt itself.
Overall, this technique is a solid first step in a text-to-SQL pipeline — not a complete solution, but a smart and efficient filtering layer.